In [4]:
import sys
print(sys.executable)

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\Scripts\python.exe


In [1]:
!py --version
!nvcc --version

Python 3.10.8
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Fri_Jun_14_16:44:19_Pacific_Daylight_Time_2024
Cuda compilation tools, release 12.6, V12.6.20
Build cuda_12.6.r12.6/compiler.34431801_0


In [1]:
!python -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -m pip install "paddleocr[all]"
!python -m pip install transformers
!python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
!python -m pip install label-studio
!python -m pip install datasets
!python -m pip install pillow
!python -m pip install jupyterlab
# python -m pip install ipykernel
!python -m pip install opencv-python
!pip install protobuf==3.20.3

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl.metadata (698 bytes)
Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl (904 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.4
    Uninstalling protobuf-6.33.4:
      Successfully uninstalled protobuf-6.33.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-core 2.30.0 requires protobuf<7.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.78.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.


In [4]:
import paddle 
import torch

OSError: [WinError 127] The specified procedure could not be found. Error loading "D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\torch\lib\cudnn_cnn64_9.dll" or one of its dependencies.

In [1]:
import json
from paddleocr import PaddleOCR
import paddle
import os
import pprint
import torch
from transformers import LayoutLMv3Tokenizer, LayoutLMv3Processor,LayoutLMv3ForTokenClassification
from PIL import Image
from torchvision import transforms
from transformers import Trainer, TrainingArguments

ModuleNotFoundError: No module named 'paddleocr'

In [3]:
!setx CUDA_HOME "C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.6"
!setx PATH "C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.6\bin;%PATH%"


SUCCESS: Specified value was saved.

SUCCESS: Specified value was saved.


# Run the code in gpu

### To check does the gpu is decteced or not 

In [ ]:
print(paddle.device.get_device())

In [ ]:
torch.cuda.is_available()

# Extracting the text and bbox from the image using paddleocr


In [ ]:
image_path = "./training data/1.jpg"
paddle_ocr_result_path = "output/output_final/paddle_ocr_result.json"

####              GPU

In [ ]:
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

print(paddle.device.get_device())
ocr = PaddleOCR(use_textline_orientation=True, lang='en', device='gpu')
result = ocr.predict(image_path)

#### Processor

In [ ]:
# os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

# ocr = PaddleOCR(
#     use_doc_orientation_classify=False, 
#     use_doc_unwarping=False, 
#     use_textline_orientation=False) # text detection + text recognition


# # image_path = "./training data/1.jpg"

# result = ocr.predict(image_path) # predict the image 

#### Dump to json file

In [ ]:
for res in result:
    res.print() # print the result of each image
    # res.save_to_img("output")
    # res.save_to_json("output/paddle_ocr_result.json")
    res.save_to_json(paddle_ocr_result_path)

# Convert ocr formate to label studio formate

In [ ]:
def convert_ocr_to_ls(ocr_json, image_url, img_w, img_h):

    results = []

    polys = ocr_json["rec_polys"]
    texts = ocr_json["rec_texts"]

    for i, poly in enumerate(polys):

        xs = [p[0] for p in poly]
        ys = [p[1] for p in poly]

        x_min = min(xs)
        y_min = min(ys)

        x_max = max(xs)
        y_max = max(ys)

        w = x_max - x_min
        h = y_max - y_min

        results.append({
            "id": f"box_{i}",
            "type": "rectangle",
            "from_name": "bbox",
            "to_name": "image",
            "original_width": img_w,
            "original_height": img_h,
            "image_rotation": 0,
            "value": {
                "x": x_min/img_w*100,
                "y": y_min/img_h*100,
                "width": w/img_w*100,
                "height": h/img_h*100,
                "rotation": 0
            }
        })
        
        # OCR Text
        results.append({
            "id": f"box_{i}",
            "type": "textarea",
            "from_name": "transcription",
            "to_name": "image",
            "value": {
                "text": [texts[i]]
            }
        })
        
    return {
        "data": {
            "ocr": image_url
        },
        "predictions":[
            {
                "model_version":"paddleocr",
                "result": results
            }
        ]
    }


In [ ]:
img_path_ls="http://localhost:8000/output/1.jpg"
ocr_to_ls_path="output/output_final/ocr_to_labelstudio.json"

In [ ]:
data=convert_ocr_to_ls(res, img_path_ls, 2232, 3300)
pprint.pprint(data)
                       
with open(ocr_to_ls_path, "w") as f:
    json.dump(data, f, indent=4)

    
    # with open("./output/paddle_ocr_result.json", "r") as f:
#     data = json.load(f)
#     print(data.get('ocr'))

# Load the labeled data of label studio

In [ ]:
label_studi_result_path="output/output_final/labelstudio_result.json"

In [ ]:
with open(label_studi_result_path, 'r') as f:
    data = json.load(f)
pprint.pprint(data)

In [ ]:
data[0]['transcription']

In [ ]:
print(data[0]['label'][0])
print(data[0]['label'][0]['x'])
print(data[0]['label'][0]['y'])
print(data[0]['label'][0]['width'])
print(data[0]['label'][0]['height'])
print(data[0]['label'][0]['original_width'])
print(data[0]['label'][0]['original_height'])

# Convert label studio formate to layoutlvm3 formate [BIO Formate]

In [ ]:
token_poly=[]
token_word=[]
word=[]
boxes=[]
bio=[]

### function

#### get min and max points 

In [ ]:
# # to get the max and min points 
# def getMinMaxPoint(poly):
#     x0 = int(poly[:,0].min())
#     y0 = int(poly[:,1].min())
#     x1 = int(poly[:,0].max())
#     y1 = int(poly[:,1].max())
#     return [x0, y0, x1, y1] 

#### tokenize the sentence and box 

In [ ]:
def tokenize_sent(xy_points, sentence_split, sentence,label_word):
    # print(len(sentence))
    letter_width=(xy_points[2]-xy_points[0])/len(sentence)
    x = xy_points[0]
    y0, y1 = xy_points[1], xy_points[3]

    for i, word in enumerate(sentence_split):
        # print(i,word)
        if label_word == "Unwanted".upper():
            bio.append('O')
        else:
            if(i==0):
                bio.append('B-'+label_word.upper())
            else:
                bio.append('I-'+label_word.upper())
        
        word_width = len(word) * letter_width
        x_end = x + word_width

        # print([int(x), y0, int(x_end), y1])
        token_word.append(word)
        token_poly.append([int(x), y0, int(x_end), y1])
        x = x_end

        # add space after word (except last)
        if i < len(sentence_split) - 1:
            x += letter_width

#### normalize points

In [ ]:
def normalize(token_ploy, image_path):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    for box in token_ploy:
        boxes.append([
            int(1000 * box[0] / width),
            int(1000 * box[1] / height),
            int(1000 * box[2] / width),
            int(1000 * box[3] / height),
            ])

#### tokenize and normalize 

In [ ]:
def tokenize_normalize(transcription,labels,token_poly,image_path):
    for sentence,label in zip(transcription,labels):
        
        x0 = label['x']
        y0 = label['y']
        x1 = x0 + label['width']
        y1 = y0 + label['height']
        width = label['original_width']
        height = label['original_height']
        label_word = label['labels'][0].upper()
        if label_word == 'Ignore'.upper():
            continue
        xy_point=[x0,y0,x1,y1]
        tokenize_sent(xy_point,sentence.split(),sentence,label_word)

    normalize(token_poly, image_path)

In [ ]:
tokenize_normalize(data[0]['transcription'],data[0]['label'],token_poly,image_path)

In [ ]:
len(token_word)

In [ ]:
len(token_poly)

In [ ]:
len(boxes)

In [ ]:
len(bio)


#### define the list for index number of bio formate

In [ ]:
label_list = {
'B-INVOICENO':0,
'B-VENDOR':1,
'B-DATE':2,
'B-PRODUCT':3,
'B-HSN/SAC':4,
'B-GST':5,
'B-QTY':6,
'B-RATE':7,
'B-AMT':8,
'B-TOTAL':9,
'B-SUBTOTAL':10,
'B-TAXTOTAL':11,

'I-INVOICENO':12,
'I-VENDOR':13,
'I-DATE':14,
'I-PRODUCT':15,
'I-HSN/SAC':16,
'I-GST':17,
'I-QTY':18,
'I-RATE':19,
'I-AMT':20,
'I-TOTAL':21,
'I-SUBTOTAL':22,
'I-TAXTOTAL':23,
'O':24}

#### assign the label index 

In [ ]:
# label_ids = [label2id[l] for l in bio]
bio_to_labelIds=[]
for bio_word in bio:
    bio_to_labelIds.append(label_list[bio_word])

# start of modle layout lvm3

#### import the image 

In [ ]:
image = Image.open(image_path).convert("RGB")

#### tokenize and convert the words to number where neural network can understand 

In [ ]:
gpu_or_cpu="cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(gpu_or_cpu) #used to set the model to run in cpu or cuda 

print("Using device:", device)
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base",apply_ocr=False)

# # Move model to GPU
# model.to(device)

In [ ]:
encoding = processor(
    images=image,               # PIL image of the document
    text=token_word,               # tokenized words from OCR
    boxes=boxes,         # bounding boxes
    word_labels=bio_to_labelIds,  # your BIO labels
    return_tensors="pt",
    padding="max_length",
    truncation=True
)

#### loading the model and  feed number of label used 

In [ ]:
num_labels = len(label_list)  # number of unique BIO labels
num_labels

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=num_labels
)

# training

In [ ]:

print(torch.version.cuda)   # the CUDA version PyTorch was built with
print(torch.backends.cudnn.version())  # optional: cuDNN version
# print(torch.cuda.get_device_name(0))

In [ ]:
torch.cuda.is_available()

In [ ]:
!nvcc --version